In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from study.data import cohort, prediction_matrix, prediction_metadata
from study.io import WORK, load, reset_directory, write_table
from study.models import apply_calibration, fit_recalibrator
from study.statistics import paired_difference

In [ ]:
reset_directory(WORK / 'predictions/calibration')

source_fits = load(WORK / 'models/source.pkl')
metadata = cohort('eicu').sort_index()
calibration_ids = metadata.index[metadata.partition.eq('calibration')]
test_ids = metadata.index[metadata.partition.eq('test')]

In [ ]:
# Combine full input and restricted predictions for each source fit
for repeat in source_fits['repeats']:
    predictions = pd.concat(
        [
            pd.read_parquet(
                WORK / 'predictions' / name,
                filters = [
                    ('repeat', '==', repeat),
                    ('domain', '==', 'eicu'),
                    ('source', '==', 'metavision'),
                    ('support', '==', 'shared'),
                    ('tier', 'in', ['R2', 'R3', 'R4']),
                    ('restriction', 'in', ['full', 'retain_0.5', 'retain_0.25', 'terminal_outage'])
                ],
            )
            for name in ('full', 'quantity', 'timing')
        ],
        ignore_index = True,
    )

    rows = []
    for tier in ('R2', 'R3', 'R4'):
        selected = predictions.loc[predictions.tier.eq(tier)]
        corrections = {('raw', 'raw'): np.array([0.0, 1.0])}

        # Reuse source corrections and fit updates on reserved calibration patients
        for method in ('intercept', 'platt'):
            corrections['source_clean', method] = source_fits['models']['metavision', repeat, 'shared', tier]['corrections'][method]

            for condition, name in [('full', 'external_clean'), ('retain_0.5', 'external_retain_0.5')]:
                calibration = selected.loc[
                    selected.restriction.eq(condition) & selected.stay_id.isin(calibration_ids)
                ].sort_values(['draw', 'stay_id'])

                corrections[name, method] = fit_recalibrator(calibration.in_hospital_mortality, calibration.risk, method)

        # Apply fitted corrections to test patients under each restriction
        for condition in ('full', 'retain_0.5', 'retain_0.25', 'terminal_outage'):
            evaluated = selected.loc[selected.restriction.eq(condition) & selected.stay_id.isin(test_ids)]

            for (calibration, method), coefficients in corrections.items():
                rows.append(
                    evaluated.assign(
                        risk = apply_calibration(coefficients, evaluated.risk),
                        calibration = calibration, method = method
                    )
                )

            write_table(
                WORK / 'predictions/calibration' / f'{tier}_{condition}_{repeat}.parquet',
                pd.concat(rows, ignore_index = True),
            )
            rows = []

    print('Calibration allocation', repeat)

In [ ]:
# Compare raw and corrected risks under each restriction
rows = []
for tier in ('R2', 'R3', 'R4'):
    for restriction in ('full', 'retain_0.5', 'retain_0.25', 'terminal_outage'):
        # Pair raw and corrected risks on the same test patients
        filters = [('tier', '==', tier), ('restriction', '==', restriction)]
        raw = prediction_matrix('calibration', test_ids, filters + [('calibration', '==', 'raw')])

        for calibration in ('source_clean', 'external_clean', 'external_retain_0.5'):
            for method in ('intercept', 'platt'):
                corrected = prediction_matrix(
                    'calibration',
                    test_ids,
                    filters + [('calibration', '==', calibration), ('method', '==', method)],
                )

                # Estimate calibration effects with patient and hospital weighting
                for equal_hospital in (False, True):
                    rows.append(
                        dict(
                            tier = tier,
                            restriction = restriction,
                            calibration = calibration,
                            method = method,
                            **paired_difference(
                                prediction_metadata(metadata.loc[test_ids]),
                                corrected,
                                raw,
                                cluster = True,
                                equal_hospital = equal_hospital,
                            ),
                        )
                    )

write_table(WORK / 'results/calibration.parquet', pd.DataFrame(rows))

display(pd.DataFrame(rows).round(5))